# Q11: Linear Regression
**Topic:** Coding-question | **Difficulty:** Easy | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
Implement Linear Regression from scratch using the Normal Equation and Gradient Descent.

## Theory

**Linear Regression** finds the best-fit line: $\hat{y} = Xw + b$

**Loss Function (MSE):** $L = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

**Method 1: Normal Equation (closed-form):**

$$w = (X^TX)^{-1}X^Ty$$

- Pros: Exact solution, no hyperparameters
- Cons: $O(n^3)$ complexity, fails if $X^TX$ is singular

**Method 2: Gradient Descent (iterative):**

$$w := w - \alpha \cdot \frac{\partial L}{\partial w}$$

where $\frac{\partial L}{\partial w} = -\frac{2}{n}X^T(y - Xw)$

- Pros: Scalable to large datasets, works for any differentiable loss
- Cons: Requires tuning learning rate $\alpha$ and iterations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class LinearRegressionNormal:
    """Linear Regression using the Normal Equation."""
    
    def __init__(self):
        self.weights: np.ndarray = None
        self.bias: float = 0.0
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegressionNormal':
        """Fit using closed-form: w = (X^T X)^{-1} X^T y."""
        # Add bias column
        X_b = np.column_stack([np.ones(len(X)), X])
        # Normal equation
        theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        self.bias = theta[0]
        self.weights = theta[1:]
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        return X @ self.weights + self.bias


class LinearRegressionGD:
    """Linear Regression using Gradient Descent."""
    
    def __init__(self, lr: float = 0.01, n_iters: int = 1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights: np.ndarray = None
        self.bias: float = 0.0
        self.loss_history: list[float] = []
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegressionGD':
        """Fit using gradient descent."""
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        
        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y
            
            # Gradients
            dw = (2 / n_samples) * (X.T @ error)
            db = (2 / n_samples) * np.sum(error)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Track loss
            mse = np.mean(error ** 2)
            self.loss_history.append(mse)
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        return X @ self.weights + self.bias

In [ ]:
# --- Generate sample data ---
np.random.seed(42)
X = 2 * np.random.rand(200, 1)
y = 4 + 3 * X.squeeze() + np.random.randn(200) * 0.5  # y = 4 + 3x + noise

# Fit both models
model_normal = LinearRegressionNormal().fit(X, y)
model_gd = LinearRegressionGD(lr=0.1, n_iters=200).fit(X, y)

print("=== Normal Equation ===")
print(f"Weights: {model_normal.weights}, Bias: {model_normal.bias:.4f}")
print(f"MSE: {np.mean((y - model_normal.predict(X))**2):.4f}")

print("\n=== Gradient Descent ===")
print(f"Weights: {model_gd.weights}, Bias: {model_gd.bias:.4f}")
print(f"MSE: {np.mean((y - model_gd.predict(X))**2):.4f}")

print(f"\nTrue: weight=3, bias=4")

In [ ]:
# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fit comparison
X_line = np.linspace(0, 2, 100).reshape(-1, 1)
axes[0].scatter(X, y, alpha=0.5, s=15, label='Data')
axes[0].plot(X_line, model_normal.predict(X_line), 'r-', linewidth=2, label='Normal Eq')
axes[0].plot(X_line, model_gd.predict(X_line), 'g--', linewidth=2, label='Gradient Descent')
axes[0].set_title('Linear Regression Fit')
axes[0].legend()
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')

# Right: GD loss curve
axes[1].plot(model_gd.loss_history)
axes[1].set_title('Gradient Descent: Loss Over Iterations')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MSE')

plt.tight_layout()
plt.show()

In [ ]:
# --- Verify against sklearn ---
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression().fit(X, y)
print(f"sklearn: weight={sk_model.coef_[0]:.4f}, bias={sk_model.intercept_:.4f}")
print(f"Ours:    weight={model_normal.weights[0]:.4f}, bias={model_normal.bias:.4f}")

## Key Takeaways
- **Normal Equation** is best for small datasets (n < 10,000 features)
- **Gradient Descent** scales to millions of samples and is the foundation of deep learning
- Always **feature scale** before GD (standardization or normalization) for faster convergence
- `np.linalg.pinv` (pseudo-inverse) is more robust than `np.linalg.inv` for singular matrices